In [1]:
import os
import glob
import pandas as pd
import pickle
import matplotlib.pyplot as plt
import numpy as np
import random
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
import pprint
import pyspark
import pyspark.sql.functions as F

from pyspark.sql.functions import col
from pyspark.sql.types import StringType, IntegerType, FloatType, DateType

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import xgboost as xgb
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import make_scorer, f1_score, roc_auc_score
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

import model_inference


In [2]:
# Build a .py script that takes a snapshot date, loads a model artefact and make an inference and save to datamart

## set up pyspark session

In [3]:
# Initialize SparkSession
spark = pyspark.sql.SparkSession.builder \
    .appName("dev") \
    .master("local[*]") \
    .getOrCreate()

# Set log level to ERROR to hide warnings
spark.sparkContext.setLogLevel("ERROR")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/10/30 04:54:31 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## set up config

In [4]:
snapshot_date_str = "2024-01-01"
model_name = "credit_model_2024_09_01.pkl"


In [5]:
config = {}
config["snapshot_date_str"] = snapshot_date_str
config["snapshot_date"] = datetime.strptime(config["snapshot_date_str"], "%Y-%m-%d")
config["model_name"] = model_name
config["model_bank_directory"] = "model_bank/"
config["model_artefact_filepath"] = config["model_bank_directory"] + config["model_name"]

pprint.pprint(config)

{'model_artefact_filepath': 'model_bank/credit_model_2024_09_01.pkl',
 'model_bank_directory': 'model_bank/',
 'model_name': 'credit_model_2024_09_01.pkl',
 'snapshot_date': datetime.datetime(2024, 1, 1, 0, 0),
 'snapshot_date_str': '2024-01-01'}


## load model artefact from model bank

In [6]:
# Load the model from the pickle file
with open(config["model_artefact_filepath"], 'rb') as file:
    model_artefact = pickle.load(file)

print("Model loaded successfully! " + config["model_artefact_filepath"])

Model loaded successfully! model_bank/credit_model_2024_09_01.pkl


## load feature store

In [7]:
# --- load feature store ---
folder_path = "datamart/gold/feature_store/"
files_list = [folder_path+os.path.basename(f) for f in glob.glob(os.path.join(folder_path, '*'))]
features_store_sdf = spark.read.option("header", "true").parquet(*files_list)
features_store_sdf = features_store_sdf.withColumnRenamed(
    "snapshot_date","feature_snapshot_date"
)

print("row_count:",features_store_sdf.count())


# extract feature store
features_sdf = features_store_sdf.filter((col("feature_snapshot_date") == config["snapshot_date"]))
print("extracted features_sdf", features_sdf.count(), config["snapshot_date"])

features_pdf = features_sdf.toPandas()
features_pdf

row_count: 8974


extracted features_sdf 485 2024-01-01 00:00:00


,customer_id,feature_snapshot_date,age,annual_income,monthly_inhand_salary,num_bank_accounts,num_credit_card,interest_rate,num_of_loan,delay_from_due_date,...,avg_fe_11,avg_fe_12,avg_fe_13,avg_fe_14,avg_fe_15,avg_fe_16,avg_fe_17,avg_fe_18,avg_fe_19,avg_fe_20
0,CUS_0x133e,2024-01-01,41,16626.250000,1096.520874,0.0,2.0,5.0,0.00000,13,...,54.000000,84.076923,87.615385,147.153846,57.307692,99.076923,93.461538,93.615385,155.076923,102.000000
1,CUS_0x14d0,2024-01-01,22,48089.160156,3902.429932,10.0,8.0,18.0,5.00000,59,...,125.538462,105.615385,59.307692,96.692308,101.538462,63.153846,95.076923,108.538462,105.461538,161.153846
2,CUS_0x1668,2024-01-01,30,64980.718750,5473.060059,4.0,7.0,6.0,2.00000,7,...,105.230769,61.846154,105.692308,90.230769,85.461538,120.307692,28.615385,87.000000,88.230769,67.307692
3,CUS_0x18cb,2024-01-01,41,47093.320312,3873.481689,3.0,6.0,18.0,6.00000,22,...,79.461538,156.153846,112.692308,75.000000,89.846154,92.769231,95.384615,104.153846,79.153846,88.538462
4,CUS_0x1eee,2024-01-01,34,115649.523438,9400.459961,5.0,5.0,9.0,2.00000,7,...,132.846154,94.461538,83.307692,86.461538,96.307692,115.769231,148.230769,50.615385,85.846154,84.153846
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
480,CUS_0xbf9c,2024-01-01,23,10628.735352,1003.727905,4.0,6.0,10.0,2.00000,10,...,105.384615,102.769231,103.230769,124.000000,122.230769,66.307692,120.769231,120.076923,52.461538,105.769231
481,CUS_0xc131,2024-01-01,44,7893.875000,900.822937,8.0,8.0,17.0,5.00000,53,...,102.923077,78.769231,125.461538,152.076923,114.692308,139.769231,53.923077,125.846154,81.846154,92.538462
482,CUS_0xc208,2024-01-01,25,26463.439453,2341.286621,3.0,4.0,16.0,2.00000,9,...,76.384615,112.615385,109.153846,115.615385,89.538462,109.538462,114.923077,78.153846,90.538462,120.769231
483,CUS_0xc50f,2024-01-01,23,38655.101562,2971.392578,5.0,7.0,5.0,4.00000,9,...,92.615385,133.384615,61.692308,105.769231,104.307692,61.307692,65.230769,130.615385,150.307692,94.846154


## preprocess data for modeling

In [8]:
# prepare X_inference
exclude_cols = [
"customer_id", "feature_snapshot_date"
]
feature_cols = [c for c in features_pdf.columns if c not in exclude_cols]
X_inference = features_pdf[feature_cols].copy()

# Identify categorical columns
cat_cols = X_inference.select_dtypes(include=['object']).columns.tolist()

# Handle categorical encoding
for col in cat_cols:
    # Try to reuse training mappings if stored; otherwise, auto-map
    try:
        mapping = model_artefact["preprocessing_transformers"].get(f"{col}_mapping", None)
        if mapping is not None:
            X_inference[col] = X_inference[col].map(mapping)
        else:
            # fallback: derive from inference data
            X_inference[col] = X_inference[col].astype('category').cat.codes
    except Exception as e:
        print(f"Warning: Could not map column {col} ({e}), fallback to category codes.")
        X_inference[col] = X_inference[col].astype('category').cat.codes

# Replace NaN / inf values
X_inference.replace([np.inf, -np.inf], np.nan, inplace=True)
X_inference.fillna(X_inference.mean(), inplace=True)

# apply transformer - standard scaler
X_inference = X_inference.astype(float)
transformer_stdscaler = model_artefact["preprocessing_transformers"]["stdscaler"]
X_inference = transformer_stdscaler.transform(X_inference)

print('X_inference', X_inference.shape[0])
X_inference

X_inference 485


array([[ 0.67774921, -0.11443755, -0.95666596, ..., -0.19987331,
         1.59137403,  0.03850144],
       [-1.08272946, -0.09302223, -0.09673261, ...,  0.23533784,
         0.12627186,  1.75437271],
       [-0.34147528, -0.08152494,  0.38462199, ..., -0.39280197,
        -0.38253882, -0.96781578],
       ...,
       [-0.80475915, -0.10774184, -0.57517976, ..., -0.65078796,
        -0.31439454,  0.58293914],
       [-0.99007269, -0.09944355, -0.38206975, ...,  0.87918115,
         1.45054251, -0.16900965],
       [-1.08272946, -0.07685185,  0.31984231, ...,  1.15960071,
         0.3897631 ,  2.42822592]], shape=(485, 72))

## model prediction inference

In [9]:
 features_pdf

,customer_id,feature_snapshot_date,age,annual_income,monthly_inhand_salary,num_bank_accounts,num_credit_card,interest_rate,num_of_loan,delay_from_due_date,...,avg_fe_11,avg_fe_12,avg_fe_13,avg_fe_14,avg_fe_15,avg_fe_16,avg_fe_17,avg_fe_18,avg_fe_19,avg_fe_20
0,CUS_0x133e,2024-01-01,41,16626.250000,1096.520874,0.0,2.0,5.0,0.00000,13,...,54.000000,84.076923,87.615385,147.153846,57.307692,99.076923,93.461538,93.615385,155.076923,102.000000
1,CUS_0x14d0,2024-01-01,22,48089.160156,3902.429932,10.0,8.0,18.0,5.00000,59,...,125.538462,105.615385,59.307692,96.692308,101.538462,63.153846,95.076923,108.538462,105.461538,161.153846
2,CUS_0x1668,2024-01-01,30,64980.718750,5473.060059,4.0,7.0,6.0,2.00000,7,...,105.230769,61.846154,105.692308,90.230769,85.461538,120.307692,28.615385,87.000000,88.230769,67.307692
3,CUS_0x18cb,2024-01-01,41,47093.320312,3873.481689,3.0,6.0,18.0,6.00000,22,...,79.461538,156.153846,112.692308,75.000000,89.846154,92.769231,95.384615,104.153846,79.153846,88.538462
4,CUS_0x1eee,2024-01-01,34,115649.523438,9400.459961,5.0,5.0,9.0,2.00000,7,...,132.846154,94.461538,83.307692,86.461538,96.307692,115.769231,148.230769,50.615385,85.846154,84.153846
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
480,CUS_0xbf9c,2024-01-01,23,10628.735352,1003.727905,4.0,6.0,10.0,2.00000,10,...,105.384615,102.769231,103.230769,124.000000,122.230769,66.307692,120.769231,120.076923,52.461538,105.769231
481,CUS_0xc131,2024-01-01,44,7893.875000,900.822937,8.0,8.0,17.0,5.00000,53,...,102.923077,78.769231,125.461538,152.076923,114.692308,139.769231,53.923077,125.846154,81.846154,92.538462
482,CUS_0xc208,2024-01-01,25,26463.439453,2341.286621,3.0,4.0,16.0,2.00000,9,...,76.384615,112.615385,109.153846,115.615385,89.538462,109.538462,114.923077,78.153846,90.538462,120.769231
483,CUS_0xc50f,2024-01-01,23,38655.101562,2971.392578,5.0,7.0,5.0,4.00000,9,...,92.615385,133.384615,61.692308,105.769231,104.307692,61.307692,65.230769,130.615385,150.307692,94.846154


In [10]:
# load model
model = model_artefact["model"]

# predict model
y_inference = model.predict_proba(X_inference)[:, 1]

# prepare output
y_inference_pdf = features_pdf[["customer_id","feature_snapshot_date"]].copy()
y_inference_pdf["model_name"] = config["model_name"]
y_inference_pdf["model_predictions"] = y_inference
y_inference_pdf

,customer_id,feature_snapshot_date,model_name,model_predictions
0,CUS_0x133e,2024-01-01,credit_model_2024_09_01.pkl,0.032150
1,CUS_0x14d0,2024-01-01,credit_model_2024_09_01.pkl,0.227160
2,CUS_0x1668,2024-01-01,credit_model_2024_09_01.pkl,0.029446
3,CUS_0x18cb,2024-01-01,credit_model_2024_09_01.pkl,0.107939
4,CUS_0x1eee,2024-01-01,credit_model_2024_09_01.pkl,0.053485
...,...,...,...,...
480,CUS_0xbf9c,2024-01-01,credit_model_2024_09_01.pkl,0.116170
481,CUS_0xc131,2024-01-01,credit_model_2024_09_01.pkl,0.930701
482,CUS_0xc208,2024-01-01,credit_model_2024_09_01.pkl,0.043068
483,CUS_0xc50f,2024-01-01,credit_model_2024_09_01.pkl,0.232881


## save model inference to datamart gold table

In [11]:
# create bronze datalake
gold_directory = f"datamart/gold/model_predictions/{config["model_name"][:-4]}/"
print(gold_directory)

if not os.path.exists(gold_directory):
    os.makedirs(gold_directory)

# save gold table - IRL connect to database to write
partition_name = config["model_name"][:-4] + "_predictions_" + snapshot_date_str.replace('-','_') + '.parquet'
filepath = gold_directory + partition_name
spark.createDataFrame(y_inference_pdf).write.mode("overwrite").parquet(filepath)
# df.toPandas().to_parquet(filepath,
#           compression='gzip')
print('saved to:', filepath)

datamart/gold/model_predictions/credit_model_2024_09_01/


saved to: datamart/gold/model_predictions/credit_model_2024_09_01/credit_model_2024_09_01_predictions_2024_01_01.parquet


## backfill

In [12]:
# set up config
snapshot_date_str = "2023-01-01"

start_date_str = "2023-01-01"
end_date_str = "2024-12-01"

In [13]:
# generate list of dates to process
def generate_first_of_month_dates(start_date_str, end_date_str):
    # Convert the date strings to datetime objects
    start_date = datetime.strptime(start_date_str, "%Y-%m-%d")
    end_date = datetime.strptime(end_date_str, "%Y-%m-%d")
    
    # List to store the first of month dates
    first_of_month_dates = []

    # Start from the first of the month of the start_date
    current_date = datetime(start_date.year, start_date.month, 1)

    while current_date <= end_date:
        # Append the date in yyyy-mm-dd format
        first_of_month_dates.append(current_date.strftime("%Y-%m-%d"))
        
        # Move to the first of the next month
        if current_date.month == 12:
            current_date = datetime(current_date.year + 1, 1, 1)
        else:
            current_date = datetime(current_date.year, current_date.month + 1, 1)

    return first_of_month_dates

dates_str_lst = generate_first_of_month_dates(start_date_str, end_date_str)


In [14]:
for snapshot_date in dates_str_lst:
    print(snapshot_date)
    model_inference.main(snapshot_date, model_name)

2023-01-01


---starting job---


{'model_artefact_filepath': 'model_bank/credit_model_2024_09_01.pkl',
 'model_bank_directory': 'model_bank/',
 'model_name': 'credit_model_2024_09_01.pkl',
 'snapshot_date': datetime.datetime(2023, 1, 1, 0, 0),
 'snapshot_date_str': '2023-01-01'}
Model loaded successfully! model_bank/credit_model_2024_09_01.pkl


row_count: 8974


extracted features_sdf 530 2023-01-01 00:00:00


X_inference 530
Sample predictions:
  customer_id feature_snapshot_date                   model_name  \
0  CUS_0x1297            2023-01-01  credit_model_2024_09_01.pkl   
1  CUS_0x13a8            2023-01-01  credit_model_2024_09_01.pkl   
2  CUS_0x1567            2023-01-01  credit_model_2024_09_01.pkl   
3  CUS_0x1733            2023-01-01  credit_model_2024_09_01.pkl   
4  CUS_0x17c7            2023-01-01  credit_model_2024_09_01.pkl   

   model_predictions  
0           0.705214  
1           0.047135  
2           0.082099  
3           0.214687  
4           0.057401  
Mean predicted probability: 0.2918
Total records: 530
datamart/gold/model_predictions/credit_model_2024_09_01/
saved to: datamart/gold/model_predictions/credit_model_2024_09_01/credit_model_2024_09_01_predictions_2023_01_01.parquet


---completed job---


2023-02-01


---starting job---


{'model_artefact_filepath': 'model_bank/credit_model_2024_09_01.pkl',
 'model_bank_directory': 'model_bank/',
 'model_name': 'c

row_count: 8974


...

extracted features_sdf 0 2024-11-01 00:00:00


No feature data found for snapshot 2024-11-01. Skipping inference.
---skipped job---


2024-12-01


---starting job---


{'model_artefact_filepath': 'model_bank/credit_model_2024_09_01.pkl',
 'model_bank_directory': 'model_bank/',
 'model_name': 'credit_model_2024_09_01.pkl',
 'snapshot_date': datetime.datetime(2024, 12, 1, 0, 0),
 'snapshot_date_str': '2024-12-01'}
Model loaded successfully! model_bank/credit_model_2024_09_01.pkl


row_count: 8974


extracted features_sdf 0 2024-12-01 00:00:00


No feature data found for snapshot 2024-12-01. Skipping inference.
---skipped job---




## Check datamart

In [15]:
# Initialize SparkSession
spark = pyspark.sql.SparkSession.builder \
    .appName("dev") \
    .master("local[*]") \
    .getOrCreate()

# Set log level to ERROR to hide warnings
spark.sparkContext.setLogLevel("ERROR")

In [16]:
folder_path = "datamart/gold/model_predictions/credit_model_2024_09_01/"
files_list = [folder_path+os.path.basename(f) for f in glob.glob(os.path.join(folder_path, '*'))]
df = spark.read.option("header", "true").parquet(*files_list)
print("row_count:",df.count())

df.show()

row_count: 62818
+-----------+---------------------+--------------------+-------------------+
|customer_id|feature_snapshot_date|          model_name|  model_predictions|
+-----------+---------------------+--------------------+-------------------+
| CUS_0xa008|                 NULL|credit_model_2024...| 0.2644983232021332|
| CUS_0xa049|                 NULL|credit_model_2024...| 0.3413495421409607|
| CUS_0xa05a|                 NULL|credit_model_2024...| 0.5226669311523438|
| CUS_0xa05f|                 NULL|credit_model_2024...| 0.3243679106235504|
| CUS_0xa08c|                 NULL|credit_model_2024...| 0.1591833084821701|
| CUS_0xa0f3|                 NULL|credit_model_2024...|0.21705250442028046|
| CUS_0xa153|                 NULL|credit_model_2024...| 0.2270229011774063|
| CUS_0xa163|                 NULL|credit_model_2024...|  0.253837913274765|
| CUS_0xa235|                 NULL|credit_model_2024...| 0.4543693959712982|
| CUS_0xa25a|                 NULL|credit_model_2024...| 0.